# Sketch 3.2 - standalone, portable notebook

Self-contained extract of **sketch_3_2_violin** from the combined *9 sketches*
notebook. The sketch code is unchanged; the shared setup it needs
(imports, data loading) is included below. Chart data is **inlined** (no external
`altair-data` files) so it renders on Linux, macOS and Windows alike.

## Shared setup *(unchanged)*

In [ ]:
import altair as alt
import pandas as pd
import numpy as np
import geopandas as gpd  # conda install -c conda-forge geopandas
import json

# Inline the chart data so every HTML output is SELF-CONTAINED and renders on
# all platforms (incl. macOS): external altair-data-*.json URL files are not
# fetched by many notebook frontends, which left the charts blank on Mac. The
# search sketches (1.13 / 2.4 / 2.6) restrict their name universe below so the
# inlined tables stay small.
alt.data_transformers.enable('default')
alt.data_transformers.disable_max_rows()
pass

In [ ]:
# Memory-lean load: another heavy job is running on this machine (~2.5 GB free),
# so we read the CSV with categorical dtypes (15k name strings stored once), derive
# the numeric year from the 122 'annais' categories instead of 3.5M strings, then
# relax categories back to plain objects (the strings stay shared) so every later
# groupby keeps its normal semantics.
raw = pd.read_csv('dpt2020.csv', sep=';',
                  dtype={'sexe': 'int8', 'preusuel': 'category',
                         'annais': 'category', 'dpt': 'category', 'nombre': 'int32'})
cat_years = pd.to_numeric(pd.Index(raw['annais'].cat.categories), errors='coerce').astype('float32')
raw['year'] = np.asarray(cat_years)[raw['annais'].cat.codes]
raw.drop(columns=['annais'], inplace=True)
raw['decade'] = (raw['year'] // 10 * 10)
raw['preusuel'] = raw['preusuel'].astype(object)  # shared strings, plain groupbys
raw['dpt'] = raw['dpt'].astype(object)

records = raw[raw.dpt != 'XX'].copy()
named = records[records.preusuel != '_PRENOMS_RARES'].copy()

dept_total = records.groupby('dpt', as_index=False)['nombre'].sum().rename(
    columns={'nombre': 'dept_total'})

ALL_NAMES = sorted(named.preusuel.unique().tolist())  # full dropdown option list
DECADES = list(range(1900, 2021, 10))
# Shared period options for charts that need an all-years aggregate (2.4).
DECADE_OPTS = [0] + DECADES
DECADE_LABELS = ['All years'] + [str(d) for d in DECADES]
SEX_OPTIONS = ['Mixed', 'Boys', 'Girls']
SEX_LABELS = ['mixed', 'boys', 'girls']

import gc
del raw
gc.collect()

print(f'{len(named):,} named rows; {len(ALL_NAMES):,} distinct names.')
named.sample(3)


## The sketch

### Sketch 3.2 - Diverging gender bars by share (decade slider)

Each bar shows the name's gender share (full width), sorted from all-male to
all-female. The decade slider recomputes up to the top 300 names and their splits.
The rank selector chooses whether the list is ranked on mixed, boys, or girls births; watch DOMINIQUE turn unisex in the 1950s-60s.


In [ ]:
def gender_table(df, label, rank_mode):
    t = (df.groupby(['preusuel', 'sexe'])['nombre'].sum()
         .unstack(fill_value=0).rename(columns={1: 'male', 2: 'female'}))
    t['total'] = t['male'] + t['female']
    if rank_mode == 'Boys':
        t['rank_metric'] = t['male']
    elif rank_mode == 'Girls':
        t['rank_metric'] = t['female']
    else:
        t['rank_metric'] = t['total']
    t = t[t['rank_metric'] > 0].nlargest(300, 'rank_metric').reset_index()
    t['prank'] = range(1, len(t) + 1)
    t['male_share'] = (t['male'] / t['total']).round(4)
    t['decade'] = label
    t['rank_mode'] = rank_mode
    return t

tabs = []
for mode in SEX_OPTIONS:
    tabs.append(gender_table(named, 0, mode))
    for d in DECADES:
        tabs.append(gender_table(named[named.decade == d], d, mode))
g32 = pd.concat(tabs, ignore_index=True)
m = g32[['decade', 'rank_mode', 'preusuel', 'male', 'total', 'male_share', 'prank']].rename(columns={'male': 'n'})
m['sexe'] = 'Male'; m['signed'] = -(m['n'] / m['total'])
f = g32[['decade', 'rank_mode', 'preusuel', 'female', 'total', 'male_share', 'prank']].rename(columns={'female': 'n'})
f['sexe'] = 'Female'; f['signed'] = f['n'] / f['total']
long32 = pd.concat([m, f], ignore_index=True)
long32['signed'] = long32['signed'].round(4)

n_names = alt.param(name='n_names', value=20,
                    bind=alt.binding_range(min=10, max=300, step=1, name='Top N names  '))
decade_32 = alt.param(name='decade_32', value=2010,
                       bind=alt.binding_range(min=1900, max=2020, step=10,
                                              name='Decade  '))
rank_mode3 = alt.param(name='rank_mode3', value='Mixed',
                       bind=alt.binding_select(options=SEX_OPTIONS, labels=SEX_LABELS,
                                               name='Rank names by  '))
sketch_3_2 = alt.Chart(long32).transform_filter(
    'datum.decade == decade_32 && datum.rank_mode == rank_mode3 && datum.prank <= n_names'
).mark_bar().encode(
    x=alt.X('signed:Q', title='100% Male  <-  share of births  ->  100% Female',
            axis=alt.Axis(format='+%'), scale=alt.Scale(domain=[-1, 1])),
    y=alt.Y('preusuel:N', title='Name',
            sort=alt.EncodingSortField(field='male_share', op='min', order='descending')),
    color=alt.Color('sexe:N', title='Sex',
                    scale=alt.Scale(domain=['Male', 'Female'], range=['#4c78a8', '#e45756'])),
    tooltip=[alt.Tooltip('preusuel:N', title='Name'), alt.Tooltip('rank_mode:N', title='Ranked by'),
             alt.Tooltip('sexe:N'), alt.Tooltip('n:Q', title='Births', format=',')],
).add_params(decade_32, n_names, rank_mode3).properties(
    width=720, height=470,
    title='3.2 - Gender split of the decade top names (slide decade, N, and ranking sex)')

sketch_3_2.save('sketch_3_2_violin.png', ppi=120)
sketch_3_2
